# 3장 · 확률의 기초 🎲

웹 챕터 **"확률의 기초"** 와 짝을 이루는 Python 실습입니다.
경우의 수로 구한 **이론 확률**이 **시뮬레이션(경험적 확률)** 과 맞아떨어지는 것을 눈으로 확인합니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch3')

## 1. 수학적 확률: 경우의 수 세기

$$P(A) = \frac{A\text{ 가 일어나는 경우의 수}}{\text{전체 경우의 수}}$$

주사위 하나로 직접 세어 봅니다. 분수 계산은 `fractions.Fraction` 이 해 줍니다.

In [ ]:
from fractions import Fraction

S = [1, 2, 3, 4, 5, 6]              # 표본공간: 나올 수 있는 모든 눈
A = [x for x in S if x % 2 == 0]    # 사건 A: 짝수

print('표본공간 S =', S)
print('사건 A(짝수) =', A)
print('P(짝수) =', Fraction(len(A), len(S)), '=', len(A) / len(S))
print('P(3의 배수) =', Fraction(len([x for x in S if x % 3 == 0]), len(S)))

## 2. 주사위 2개: 합의 이론 분포

두 주사위 눈의 조합은 6 × 6 = 36가지. 합마다 경우의 수를 전부 세면
**합 7이 6가지로 가장 많습니다** (그래서 확률도 6/36으로 최고).

In [ ]:
from collections import Counter

ways = Counter()
for a in range(1, 7):
    for b in range(1, 7):
        ways[a + b] += 1

print('합   경우의 수   이론 확률')
for s in range(2, 13):
    print(f'{s:2d}    {ways[s]}가지      {ways[s]}/36 = {ways[s]/36:.3f}')

## 3. 10만 번 굴려 보기: 경험적 확률

이론은 이론이고, 진짜 굴리면 그렇게 나올까요? NumPy 로 주사위 2개를 **10만 번** 굴려
상대도수를 이론값과 겹쳐 그립니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'    # 한글 폰트 (Windows)
plt.rcParams['axes.unicode_minus'] = False

rng = np.random.default_rng(42)                  # 시드 고정 — 결과 재현 가능

N = 100_000
dice = rng.integers(1, 7, size=(N, 2))           # 주사위 2개를 N번 굴린 표
sums = dice.sum(axis=1)                          # 각 판의 합

observed = np.bincount(sums, minlength=13)[2:] / N          # 합 2~12의 상대도수
theory = np.array([ways[s] for s in range(2, 13)]) / 36     # 이론 확률

x = np.arange(2, 13)
plt.figure(figsize=(8, 4))
plt.bar(x - 0.2, observed, width=0.4, label=f'시뮬레이션 ({N:,}번)', color='#6366f1')
plt.bar(x + 0.2, theory, width=0.4, label='이론 (경우의 수/36)', color='#f43f5e')
plt.xticks(x)
plt.xlabel('두 주사위의 합')
plt.ylabel('비율')
plt.title('주사위 2개의 합: 시뮬레이션 vs 이론 (삼각형 모양!)')
plt.legend()
plt.show()

## 4. 던질수록 이론값에 붙는다: 큰 수의 법칙 맛보기

"합 = 7" 의 비율이 던진 횟수에 따라 어떻게 변하는지 봅니다.
처음엔 요동치지만 횟수가 늘수록 이론값 6/36 ≈ 0.167 에 착 달라붙어요.
(이 성질의 정식 이름이 **큰 수의 법칙**이고, 7장에서 제대로 다룹니다.)

In [ ]:
is7 = (sums == 7).astype(int)
running = np.cumsum(is7) / np.arange(1, N + 1)   # 누적 비율

plt.figure(figsize=(8, 3.5))
plt.plot(running, color='#6366f1', linewidth=1)
plt.axhline(6 / 36, color='#f43f5e', linestyle='--', label='이론값 6/36 ≈ 0.167')
plt.xscale('log')
plt.xlabel('던진 횟수 (로그 눈금)')
plt.ylabel('합=7 이 나온 비율')
plt.title('던질수록 이론 확률에 수렴')
plt.legend()
plt.show()

## 5. '적어도 한 번 6': 여사건 트릭 검증

웹 챕터에서 계산한 값:

$$P(\text{적어도 한 번 } 6) = 1 - \left(\tfrac{5}{6}\right)^2 = \tfrac{11}{36} \approx 0.306$$

시뮬레이션이 정말 이 값을 주는지 확인합니다.

In [ ]:
at_least_one_6 = (dice == 6).any(axis=1).mean()   # 한 판에서 6이 하나라도 있으면 True
print('시뮬레이션 :', round(at_least_one_6, 4))
print('이론 11/36 :', round(11 / 36, 4))

## 6. 덧셈법칙·곱셈법칙 검증

- **덧셈법칙(배반)**: 합이 2 이면서 동시에 12일 수는 없으니 P(2 또는 12) = 1/36 + 1/36
- **곱셈법칙(독립)**: 첫째 주사위와 둘째 주사위는 서로 영향이 없으니 P(둘 다 6) = 1/6 × 1/6

In [ ]:
p_or = ((sums == 2) | (sums == 12)).mean()
print('P(합=2 또는 12)  시뮬레이션:', round(p_or, 4), ' / 이론 2/36 :', round(2 / 36, 4))

p_and = ((dice[:, 0] == 6) & (dice[:, 1] == 6)).mean()
print('P(둘 다 6)      시뮬레이션:', round(p_and, 4), ' / 이론 1/36 :', round(1 / 36, 4))

## 정리 🎯

- **확률 = 경우의 수의 비율** (등확률일 때). 시뮬레이션 비율(경험적 확률)은 시행이 많을수록 이론값에 수렴.
- 주사위 2개의 합은 **7이 최다** (6/36). 경우의 수가 만드는 삼각형 분포.
- "또는" → **덧셈**(배반일 때), "그리고" → **곱셈**(독립일 때), "적어도 하나" → **1 − 전부 아님**.
- AI 연결: 언어모델의 출력도 결국 **모든 후보 단어에 대한 확률** (합 = 1)이고, 다음 챕터들에서 이 연결이 계속 깊어집니다.

다음 노트북 `ch04_bayes.ipynb`: 정보가 확률을 바꾸는 규칙, 베이즈 정리.